# SoccerNet-MVFoul — Dive / Simulation Summary Statistics

**Project:** Computer Vision-Assisted VAR — Contact-Response Hypothesis  
**Dataset:** SoccerNet-MVFoul (Held et al., CVPR Workshop 2023)  
**Goal:** Understand the distribution of `Dive/Simulation` actions relative to all foul types, severity, contact, and body-part annotations — before any model training.

---
### What this notebook does
1. Installs the `SoccerNet` package and downloads the MVFoul annotation JSON files  
2. Parses every action across train / valid / test splits into a single flat `DataFrame`  
3. Generates dataset-wide summary statistics with special focus on `Dive/Simulation`  
4. Plots distributions for foul type, severity, offence label, contact, and body part  
5. Computes a cross-tab of *Dive* vs every other annotation dimension  
6. Identifies the hardest confusable class pairs (most likely to be misclassified)  

> **Note:** Only the annotation JSONs are downloaded here — not the video clips (which are ~700 GB). Video download requires a free SoccerNet account password; instructions are in Cell 3.

## Cell 1 — Install dependencies

In [ ]:
# ── Install ──────────────────────────────────────────────────────────────────
!pip install SoccerNet --quiet
!pip install matplotlib seaborn pandas numpy --quiet

print('✓ Dependencies installed')

## Cell 2 — Imports and display settings

In [ ]:
import json
import os
import zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import SoccerNet
from SoccerNet.Downloader import SoccerNetDownloader

# ── Plotting style ────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
})

# ── Color palette — consistent across all plots ───────────────────────────────
DIVE_COLOR   = '#E24B4A'   # red   — dive/simulation (highlight)
OTHER_COLOR  = '#378ADD'   # blue  — other foul types
CARD_COLORS  = {
    'No card':     '#3B8BD4',
    'Yellow card': '#EF9F27',
    'Red card':    '#E24B4A',
    'No card/Yellow card': '#9FE1CB',
    'Yellow card/Red card': '#F0997B',
}

DATA_DIR = Path('/content/mvfoul_data')
DATA_DIR.mkdir(exist_ok=True)

print('✓ Imports complete')
print(f'  SoccerNet version: {SoccerNet.__version__}')

## Cell 3 — Download MVFoul annotation JSONs

The annotations are **public** (no password required for JSON-only download).  
Video clips require a free account at https://www.soccer-net.org — set `password` below if you have one.

In [ ]:
# ── Download settings ─────────────────────────────────────────────────────────
# Annotations are public — leave password as-is for JSON-only download.
# To also download video clips, register at soccer-net.org and set your password.
SOCCERNET_PASSWORD = 'password'   # <-- replace if you have a SoccerNet account

mySoccerNetDownloader = SoccerNetDownloader(LocalDirectory=str(DATA_DIR))
mySoccerNetDownloader.password = SOCCERNET_PASSWORD

# Download the MVFoul annotation zips (small, ~1 MB each)
# NOTE: task='mvfouls' downloads the 224p version (annotations + video clips)
# The SoccerNet downloader does NOT auto-extract — we unzip manually below.
mySoccerNetDownloader.downloadDataTask(
    task='mvfouls',
    split=['train', 'valid', 'test', 'challenge'],
    password=SOCCERNET_PASSWORD,
    verbose=True
)

# ── Unzip all downloaded archives ─────────────────────────────────────────────
import zipfile
MVFOUL_DIR = DATA_DIR / 'mvfouls'

for zip_path in sorted(MVFOUL_DIR.glob('*.zip')):
    split_name = zip_path.stem.replace('_720p', '')  # handle 720p variant names
    out_dir = MVFOUL_DIR / split_name
    if out_dir.exists():
        print(f'  {zip_path.name} already extracted → {out_dir.name}/')
        continue
    print(f'  Extracting {zip_path.name} ...', end=' ')
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(out_dir)
    print('done')

# ── List what we have ─────────────────────────────────────────────────────────
print('\n── File tree after extraction ──')
for p in sorted(MVFOUL_DIR.rglob('*.json')):
    print(f'  {p.relative_to(DATA_DIR)}  ({p.stat().st_size / 1024:.1f} KB)')


## Cell 4 — Parse annotations into a flat DataFrame

The MVFoul JSON has this structure:
```
{
  "Set": {
    "<action_id>": {
      "clips": [ { "path": "...", "clip_start": ..., "clip_stop": ... }, ... ],
      "Offence":       "Offence" | "No offence" | "Between",
      "Severity":      "No card" | "Yellow card" | "Red card" | ...,
      "Action class":  "Standing tackling" | "Tackling" | "High leg" | "Pushing" |
                       "Holding" | "Elbowing" | "Challenge" | "Dive/Simulation",
      "Bodypart":      "Upper body" | "Under body",
      "Contact":       "With contact" | "Without contact",
      "handball":      "No handball" | "Handball",
      "handball_offence": "No handball offence" | "Handball offence",
      "trytoplay":     "Try to play" | "No try to play",
      "touchball":     "Touch ball" | "No touch ball",
      "upper_body_part": "Use of shoulder" | "Use of arm" | ...
    }
  }
}
```

In [ ]:
# ── Annotation keys present in the MVFoul JSON ────────────────────────────────
# Schema confirmed from SoccerNet/Evaluation/MV_FoulRecognition.py:
#   Top-level key : 'Actions'  (NOT 'Set')
#   Action class  : 'Dive' (NOT 'Dive/Simulation')
#   Severity      : numeric strings '1.0'=No card  '3.0'=Yellow  '5.0'=Red
#   Offence       : 'Offence' | 'No offence' | 'Between'
ANNOTATION_KEYS = [
    'Offence', 'Severity', 'Action class', 'Bodypart', 'Contact',
    'handball', 'handball_offence', 'trytoplay', 'touchball', 'upper_body_part'
]

# Severity numeric codes → human-readable labels
SEVERITY_MAP = {
    '1.0': 'No card',
    '3.0': 'Yellow card',
    '5.0': 'Red card',
    '2.0': 'No card/Yellow card',   # borderline
    '4.0': 'Yellow card/Red card',  # borderline
    '':    'Unknown',
}


def load_split(json_path: Path, split_name: str) -> list[dict]:
    """Parse one MVFoul annotation JSON into a list of flat dicts."""
    with open(json_path) as f:
        raw = json.load(f)

    # Top-level key is 'Actions' in the official schema
    actions = raw.get('Actions', raw.get('Set', {}))
    if not actions:
        print(f'  WARNING: no Actions found in {json_path.name}. Keys: {list(raw.keys())}')
        return []

    rows = []
    for action_id, data in actions.items():
        clips = data.get('clips', [])
        severity_raw = str(data.get('Severity', '')).strip()
        row = {
            'action_id':      action_id,
            'split':          split_name,
            'n_clips':        len(clips),
            'has_replay':     any(
                'replay' in c.get('path', '').lower() for c in clips
            ),
            'severity_raw':   severity_raw,
            'Severity':       SEVERITY_MAP.get(severity_raw, severity_raw),
        }
        for k in ANNOTATION_KEYS:
            if k == 'Severity':
                continue  # already mapped above
            row[k] = str(data.get(k, 'Unknown')).strip()
        rows.append(row)
    return rows


# ── Discover annotation JSONs (inside extracted split folders) ────────────────
# Layout after extraction:
#   DATA_DIR/mvfouls/train/      <- extracted from train.zip
#     annotations.json           <- or possibly: train/train/annotations.json
#   DATA_DIR/mvfouls/valid/
#     annotations.json

MVFOUL_DIR = DATA_DIR / 'mvfouls'
split_map = {}

for json_path in sorted(MVFOUL_DIR.rglob('*.json')):
    path_str = str(json_path).lower()
    for s in ['train', 'valid', 'test', 'challenge']:
        if s in path_str and s not in split_map:
            split_map[s] = json_path
            break

print('Splits found:', list(split_map.keys()))
if not split_map:
    print('\n⚠ No JSON files found. Check that Cell 3 ran without errors.')
    print('   Expected layout: DATA_DIR/mvfouls/<split>/<anything>.json')
    print('   Actual contents of DATA_DIR/mvfouls:')
    for p in sorted(MVFOUL_DIR.rglob('*')):
        print(f'     {p.relative_to(MVFOUL_DIR)}')

# ── Load all splits ────────────────────────────────────────────────────────────
all_rows = []
for split_name, json_path in split_map.items():
    rows = load_split(json_path, split_name)
    all_rows.extend(rows)
    print(f'  {split_name:12s} → {len(rows):5d} actions  (from {json_path.name})')

if not all_rows:
    raise RuntimeError('No actions loaded — see warnings above.')

df = pd.DataFrame(all_rows)

# ── is_dive flag: action class is 'Dive' (confirmed from evaluation code) ──────
df['is_dive'] = df['Action class'].str.lower().str.contains('dive', na=False)
df['Action class'] = df['Action class'].str.strip()

print(f'\nTotal actions loaded: {len(df):,}')
print(f'Columns: {list(df.columns)}')
df.head(3)


## Cell 5 — Dataset-wide summary statistics

In [ ]:
print('=' * 60)
print('SOCCERNET-MVFOUL — DATASET SUMMARY')
print('=' * 60)

# ── Split counts ──────────────────────────────────────────────────────────────
print('\n── Actions per split ──')
print(df.groupby('split').size().rename('count').to_string())

# ── Action class distribution ─────────────────────────────────────────────────
print('\n── Action class distribution (all splits) ──')
class_counts = df['Action class'].value_counts()
class_pct    = (class_counts / len(df) * 100).round(2)
class_df     = pd.DataFrame({'count': class_counts, 'pct': class_pct})
class_df.columns = ['count', '%']
print(class_df.to_string())

# ── Dive/Simulation breakdown ─────────────────────────────────────────────────
n_dive     = df['is_dive'].sum()
n_total    = len(df)
n_non_dive = n_total - n_dive

print(f'\n── Dive / Simulation ──')
print(f'  Dive/Simulation actions : {n_dive:5d}  ({n_dive/n_total*100:.2f}% of dataset)')
print(f'  All other fouls         : {n_non_dive:5d}  ({n_non_dive/n_total*100:.2f}%)')
print(f'  Class imbalance ratio   : 1 : {n_non_dive/max(n_dive,1):.0f}')

# ── Dive per split ────────────────────────────────────────────────────────────
print('\n── Dive actions per split ──')
print(
    df.groupby('split')['is_dive']
      .agg(['sum', 'mean'])
      .rename(columns={'sum': 'dive_count', 'mean': 'dive_rate'})
      .assign(dive_rate=lambda x: (x['dive_rate'] * 100).round(2))
      .to_string()
)

## Cell 6 — Plot 1: Foul type distribution (dive highlighted)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

class_order = df['Action class'].value_counts().index.tolist()
counts      = df['Action class'].value_counts()[class_order]
colors      = [DIVE_COLOR if 'Dive' in c or 'Simulation' in c else OTHER_COLOR
               for c in class_order]

bars = ax.barh(class_order[::-1], counts[class_order[::-1]].values,
               color=colors[::-1], edgecolor='white', linewidth=0.5)

# Percentage labels
for bar, count in zip(bars, counts[class_order[::-1]].values):
    pct = count / len(df) * 100
    ax.text(bar.get_width() + 8, bar.get_y() + bar.get_height() / 2,
            f'{count:,}  ({pct:.1f}%)',
            va='center', fontsize=10, color='#444')

ax.set_xlabel('Number of actions', labelpad=8)
ax.set_title('MVFoul — Action class distribution (Dive/Simulation highlighted)')
ax.set_xlim(0, counts.max() * 1.35)
ax.tick_params(axis='y', labelsize=11)

legend_patches = [
    mpatches.Patch(color=OTHER_COLOR, label='Other foul types'),
    mpatches.Patch(color=DIVE_COLOR,  label='Dive / Simulation'),
]
ax.legend(handles=legend_patches, loc='lower right', frameon=False)
plt.tight_layout()
plt.savefig('/content/plot1_foul_distribution.png', bbox_inches='tight')
plt.show()
print('Saved → /content/plot1_foul_distribution.png')

## Cell 7 — Plot 2: Severity distribution — Dive vs. everything else

In [ ]:
severity_order = ['No card', 'No card/Yellow card', 'Yellow card',
                  'Yellow card/Red card', 'Red card']

def severity_dist(sub_df):
    counts = sub_df['Severity'].value_counts()
    total  = len(sub_df)
    return {s: counts.get(s, 0) / total * 100 for s in severity_order}

dive_sev     = severity_dist(df[df['is_dive']])
nondive_sev  = severity_dist(df[~df['is_dive']])

x    = np.arange(len(severity_order))
w    = 0.35
fig, ax = plt.subplots(figsize=(10, 5))

b1 = ax.bar(x - w/2, [nondive_sev[s] for s in severity_order],
            w, label='Other fouls', color=OTHER_COLOR, alpha=0.88)
b2 = ax.bar(x + w/2, [dive_sev[s]    for s in severity_order],
            w, label='Dive / Simulation', color=DIVE_COLOR, alpha=0.88)

for bars in (b1, b2):
    for bar in bars:
        h = bar.get_height()
        if h > 0.5:
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.5,
                    f'{h:.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(severity_order, rotation=20, ha='right')
ax.set_ylabel('% of subgroup')
ax.set_title('Severity distribution — Dive/Simulation vs. other fouls')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('/content/plot2_severity_comparison.png', bbox_inches='tight')
plt.show()
print('Saved → /content/plot2_severity_comparison.png')

## Cell 8 — Plot 3: Contact analysis
The contact/no-contact split is the most direct signal for the contact-response hypothesis.

In [ ]:
contact_cross = (
    df.groupby(['is_dive', 'Contact'])
      .size()
      .rename('count')
      .reset_index()
)
contact_cross['label'] = contact_cross['is_dive'].map(
    {True: 'Dive/Simulation', False: 'Other fouls'}
)

# Within-group percentages
totals = contact_cross.groupby('label')['count'].transform('sum')
contact_cross['pct'] = contact_cross['count'] / totals * 100

print('── Contact breakdown ──')
print(contact_cross.pivot(index='label', columns='Contact', values='pct').round(1).to_string())

# ── Plot ──────────────────────────────────────────────────────────────────────
pivot  = contact_cross.pivot(index='label', columns='Contact', values='pct').fillna(0)
colors = ['#5DCAA5', '#D85A30']   # teal = with contact, coral = without

fig, ax = plt.subplots(figsize=(7, 4))
pivot.plot(kind='bar', ax=ax, color=colors, edgecolor='white', linewidth=0.5,
           rot=0, width=0.55)

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=9)

ax.set_xlabel('')
ax.set_ylabel('% of subgroup')
ax.set_title('Contact rate — Dive/Simulation vs. other fouls\n'
             '(key signal for contact-response hypothesis)')
ax.legend(title='Contact', frameon=False, bbox_to_anchor=(1, 1))
ax.set_ylim(0, 115)
plt.tight_layout()
plt.savefig('/content/plot3_contact_analysis.png', bbox_inches='tight')
plt.show()

## Cell 9 — Plot 4: Body part and upper-body sub-type for dives

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Body part (upper vs under) ────────────────────────────────────────────────
bp_cross = (
    df.groupby(['is_dive', 'Bodypart'])
      .size()
      .rename('count')
      .reset_index()
)
bp_cross['label'] = bp_cross['is_dive'].map({True: 'Dive', False: 'Other'})
bp_pivot = bp_cross.pivot(index='label', columns='Bodypart', values='count').fillna(0)
bp_pivot = bp_pivot.div(bp_pivot.sum(axis=1), axis=0) * 100

bp_pivot.plot(kind='bar', ax=axes[0], color=['#378ADD', '#BA7517'],
              edgecolor='white', rot=0, width=0.5)
for c in axes[0].containers:
    axes[0].bar_label(c, fmt='%.1f%%', padding=2, fontsize=9)
axes[0].set_ylim(0, 120)
axes[0].set_title('Body part involved')
axes[0].set_xlabel('')
axes[0].set_ylabel('% of subgroup')
axes[0].legend(title='Body part', frameon=False)

# ── Upper-body sub-type (only for dives with upper body) ──────────────────────
dive_upper = df[(df['is_dive']) & (df['Bodypart'] == 'Upper body')]
if len(dive_upper) > 0 and 'upper_body_part' in dive_upper.columns:
    ub_counts = dive_upper['upper_body_part'].value_counts()
    colors_ub = sns.color_palette('Blues_d', len(ub_counts))[::-1]
    ub_counts.plot(kind='barh', ax=axes[1], color=colors_ub, edgecolor='white')
    for i, v in enumerate(ub_counts):
        axes[1].text(v + 0.2, i, f'{v}', va='center', fontsize=9)
    axes[1].set_title('Upper-body sub-type\n(Dive/Simulation, upper body only)')
    axes[1].set_xlabel('Count')
else:
    axes[1].text(0.5, 0.5, 'No upper-body sub-type data\nfor Dive/Simulation',
                 ha='center', va='center', transform=axes[1].transAxes, fontsize=11)
    axes[1].set_title('Upper-body sub-type (Dive only)')

plt.suptitle('Body part analysis — Dive/Simulation vs. other fouls', y=1.02)
plt.tight_layout()
plt.savefig('/content/plot4_bodypart.png', bbox_inches='tight')
plt.show()

## Cell 10 — Plot 5: "Try to play" and "Touch ball" for dives
Genuine fouls usually involve a player attempting to play the ball. Dives typically do not.

In [ ]:
intent_cols = ['trytoplay', 'touchball']
intent_labels = {'trytoplay': 'Tried to play ball', 'touchball': 'Touched ball'}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, intent_cols):
    if col not in df.columns:
        ax.text(0.5, 0.5, f'Column "{col}" not in dataset',
                ha='center', va='center', transform=ax.transAxes)
        continue

    cross = (
        df.groupby(['is_dive', col])
          .size()
          .rename('count')
          .reset_index()
    )
    cross['label'] = cross['is_dive'].map({True: 'Dive', False: 'Other'})
    pivot = cross.pivot(index='label', columns=col, values='count').fillna(0)
    pivot = pivot.div(pivot.sum(axis=1), axis=0) * 100

    palette = sns.color_palette('Set2', pivot.shape[1])
    pivot.plot(kind='bar', ax=ax, color=palette, edgecolor='white', rot=0, width=0.5)
    for c in ax.containers:
        ax.bar_label(c, fmt='%.1f%%', padding=2, fontsize=9)
    ax.set_ylim(0, 120)
    ax.set_title(intent_labels[col])
    ax.set_xlabel('')
    ax.set_ylabel('% of subgroup')
    ax.legend(frameon=False)

plt.suptitle('Intent signals — Dive/Simulation vs. other fouls\n'
             '(supports contact-response hypothesis)', y=1.02)
plt.tight_layout()
plt.savefig('/content/plot5_intent_signals.png', bbox_inches='tight')
plt.show()

## Cell 11 — Plot 6: Full cross-tab heatmap (Dive vs all annotation dimensions)

In [ ]:
# Build a summary row-per-feature showing dive_rate for each value
cross_rows = []
feature_cols = ['Offence', 'Severity', 'Bodypart', 'Contact',
                'trytoplay', 'touchball', 'handball']

for col in feature_cols:
    if col not in df.columns:
        continue
    for val, sub in df.groupby(col):
        dive_rate = sub['is_dive'].mean() * 100
        cross_rows.append({
            'feature':    col,
            'value':      str(val),
            'n':          len(sub),
            'n_dive':     int(sub['is_dive'].sum()),
            'dive_rate':  round(dive_rate, 2)
        })

cross_df = pd.DataFrame(cross_rows)

# ── Heatmap ───────────────────────────────────────────────────────────────────
pivot = cross_df.pivot(index='value', columns='feature', values='dive_rate').fillna(0)

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(
    pivot,
    annot=True, fmt='.1f', cmap='YlOrRd',
    linewidths=0.4, linecolor='white',
    cbar_kws={'label': 'Dive/Simulation rate (%)', 'shrink': 0.7},
    ax=ax
)
ax.set_title('Dive/Simulation rate (%) within each annotation category\n'
             '(higher = this value more often co-occurs with a dive)',
             pad=12)
ax.set_xlabel('Annotation feature')
ax.set_ylabel('Annotation value')
plt.tight_layout()
plt.savefig('/content/plot6_dive_crosstab_heatmap.png', bbox_inches='tight')
plt.show()

print('\n── Top co-occurring values with Dive/Simulation ──')
print(cross_df.sort_values('dive_rate', ascending=False).head(10).to_string(index=False))

## Cell 12 — Hardest confusable class pairs
Which non-dive foul class is most likely to look like a dive (based on annotation similarity)?

In [ ]:
# For each action class, compute the % with "Without contact" AND "No offence" —
# the two annotations that most resemble a simulation
confusable_features = {
    'no_contact':  df['Contact'] == 'Without contact',
    'no_offence':  df['Offence'] == 'No offence',
    'no_ball':     df['touchball'].str.contains('No touch', na=False),
    'no_try':      df['trytoplay'].str.contains('No try', na=False),
}

confusion_rows = []
for foul_class, sub in df.groupby('Action class'):
    row = {'foul_class': foul_class, 'n': len(sub)}
    for feat, mask in confusable_features.items():
        row[feat + '_pct'] = (sub.index.isin(df[mask].index)).sum() / len(sub) * 100
    # composite "dive-like" score: average across features
    row['dive_like_score'] = np.mean([row[f + '_pct'] for f in confusable_features])
    confusion_rows.append(row)

conf_df = pd.DataFrame(confusion_rows).sort_values('dive_like_score', ascending=False)

print('── Dive-like annotation profile per foul class ──')
print('(higher score = more annotations that superficially resemble a dive)\n')
display_cols = ['foul_class', 'n', 'no_contact_pct', 'no_offence_pct',
                'no_ball_pct', 'no_try_pct', 'dive_like_score']
print(conf_df[display_cols].to_string(index=False, float_format='{:.1f}'.format))

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
palette = [DIVE_COLOR if 'Dive' in c else OTHER_COLOR
           for c in conf_df['foul_class']]

ax.barh(conf_df['foul_class'][::-1], conf_df['dive_like_score'][::-1],
        color=palette[::-1], edgecolor='white')
ax.set_xlabel('Composite dive-like score (avg of 4 annotation features, %)')
ax.set_title('Which foul classes most resemble a dive in annotation space?\n'
             '(confusable pairs to watch during training)')
ax.axvline(conf_df[conf_df['foul_class'].str.contains('Dive', na=False)]
           ['dive_like_score'].mean(),
           color=DIVE_COLOR, linestyle='--', linewidth=1.2, alpha=0.6,
           label='Dive/Simulation mean')
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.savefig('/content/plot7_confusable_classes.png', bbox_inches='tight')
plt.show()

## Cell 13 — Class imbalance: what weight to use in training loss?
Computes inverse-frequency class weights for `torch.nn.CrossEntropyLoss(weight=...)`.

In [ ]:
# Limit to train split for weight calculation
train_df = df[df['split'] == 'train'].copy()
class_counts_train = train_df['Action class'].value_counts()

# Inverse-frequency weights (normalised so mean = 1)
inv_freq = 1.0 / class_counts_train
weights  = inv_freq / inv_freq.mean()

print('── Inverse-frequency class weights (for weighted loss) ──')
print('Paste these into your training script:\n')
print('class_weights = {')
for cls, w in weights.sort_values(ascending=False).items():
    flag = '  # <-- Dive: highest weight' if 'Dive' in cls else ''
    print(f'    "{cls}": {w:.4f},{flag}')
print('}')

# ── Visualise ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

colors = [DIVE_COLOR if 'Dive' in c else OTHER_COLOR for c in class_counts_train.index]
class_counts_train.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white', rot=35)
axes[0].set_title('Training set: raw class counts')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', labelsize=9)

w_colors = [DIVE_COLOR if 'Dive' in c else OTHER_COLOR for c in weights.index]
weights.sort_values(ascending=False).plot(
    kind='bar', ax=axes[1], color=w_colors, edgecolor='white', rot=35)
axes[1].set_title('Inverse-frequency training weights')
axes[1].set_ylabel('Weight (normalised, mean = 1)')
axes[1].axhline(1.0, color='gray', linestyle='--', linewidth=0.8, label='mean = 1')
axes[1].legend(frameon=False, fontsize=9)
axes[1].tick_params(axis='x', labelsize=9)

plt.suptitle('Class imbalance and recommended loss weights', y=1.02)
plt.tight_layout()
plt.savefig('/content/plot8_class_weights.png', bbox_inches='tight')
plt.show()

## Cell 14 — Export summary tables to CSV

In [ ]:
# ── Full flattened DataFrame ──────────────────────────────────────────────────
df.to_csv('/content/mvfoul_all_actions.csv', index=False)
print(f'Saved all {len(df):,} actions → /content/mvfoul_all_actions.csv')

# ── Dive-only subset ──────────────────────────────────────────────────────────
df[df['is_dive']].to_csv('/content/mvfoul_dives_only.csv', index=False)
print(f'Saved {df["is_dive"].sum():,} dive actions → /content/mvfoul_dives_only.csv')

# ── Class-weight table ────────────────────────────────────────────────────────
weights.rename('weight').to_csv('/content/class_weights.csv')
print('Saved class weights → /content/class_weights.csv')

# ── Cross-tab summary ─────────────────────────────────────────────────────────
cross_df.to_csv('/content/dive_crosstab.csv', index=False)
print('Saved cross-tab     → /content/dive_crosstab.csv')

print('\n── File summary ──────────────────────────────────────────')
for f in ['/content/mvfoul_all_actions.csv', '/content/mvfoul_dives_only.csv',
          '/content/class_weights.csv', '/content/dive_crosstab.csv']:
    size_kb = Path(f).stat().st_size / 1024
    print(f'  {f}  ({size_kb:.1f} KB)')

## Cell 15 — Quick sanity check: does everything line up with the VARS paper?

Table 2 from Held et al. (2023) reports these proportions — let's verify our parse matches.

In [ ]:
# Values from Held et al. (2023) Table 2 — using the evaluation code's class names
# Note: paper uses 'Dive/Simulation' in text but evaluation code uses 'Dive'
paper_values = {
    'Standing tackling': 43.6,
    'Tackling':          15.6,
    'Challenge':         13.0,
    'Holding':           12.5,
    'Elbowing':           5.9,
    'High leg':           3.5,
    'Pushing':            2.9,
    'Dive':               0.9,
}

print('── Sanity check: our parse vs. VARS paper (Table 2) ──')
print(f'{"Action class":<26} {"Paper %":>10} {"Our %":>8} {"Diff":>8}')
print('-' * 56)
our_pcts = (df['Action class'].value_counts() / len(df) * 100)
print('Action classes found in data:', list(our_pcts.index))
print()
for cls, paper_pct in sorted(paper_values.items(), key=lambda x: -x[1]):
    our_pct = our_pcts.get(cls, 0)
    diff    = our_pct - paper_pct
    flag    = '  ← check' if abs(diff) > 3 else ''
    print(f'{cls:<26} {paper_pct:>10.1f} {our_pct:>8.1f} {diff:>+8.1f}{flag}')

print('\n(Differences expected if using full dataset vs. paper train split only.)')
print('If "check" flags appear, re-examine the JSON parse in Cell 4.')


---
## Summary of key findings

Run all cells above, then review this cell to get a one-paragraph summary of what the data told you.

In [ ]:
n_total    = len(df)
n_dive     = df['is_dive'].sum()
dive_pct   = n_dive / n_total * 100

# Most common severity for dives
dive_sev_mode = df[df['is_dive']]['Severity'].mode()[0] if n_dive > 0 else 'N/A'

# Contact rate for dives vs others
dive_contact_pct  = (df[df['is_dive']]['Contact'] == 'With contact').mean() * 100
other_contact_pct = (df[~df['is_dive']]['Contact'] == 'With contact').mean() * 100

summary = f"""
DATASET SUMMARY — SoccerNet-MVFoul
===================================
Total actions      : {n_total:,}
Dive/Simulation    : {n_dive:,}  ({dive_pct:.2f}% — severely imbalanced)

Class imbalance ratio      : 1 dive per {n_total//max(n_dive,1)} other actions
Recommended training weight: ~{n_total/max(n_dive,1):.0f}x for Dive/Simulation

Most common severity for dives : {dive_sev_mode}
Contact rate — dives           : {dive_contact_pct:.1f}%
Contact rate — other fouls     : {other_contact_pct:.1f}%
→ Contact signal separability  : {abs(dive_contact_pct - other_contact_pct):.1f} pp difference

Implication for the project:
  The contact-response hypothesis is supported — dives show a distinctly
  lower "With contact" rate than genuine fouls. However, the class imbalance
  is severe and weighted loss + balanced accuracy (not raw accuracy) must
  be used as the primary training and evaluation metric.
"""
print(summary)